In [1]:
import os

In [2]:
%pwd

'c:\\Users\\USER\\DataScience\\AWS-Deployment\\Image-Classification\\CT-Scan-Lung-Cancer-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\USER\\DataScience\\AWS-Deployment\\Image-Classification\\CT-Scan-Lung-Cancer-Classification'

In [5]:
# os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/MannShrestha/CT-Scan-Lung-Cancer-Classification.mlflow"
# os.environ["MLFLOW_TRACKING_USERNAME"]="MannShrestha"
# os.environ["MLFLOW_TRACKING_PASSWORD"]="6824692c47a35333f9eac5b10041d5c8edbcef0"

In [6]:
import tensorflow as tf

In [7]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

## Config Entity

In [9]:

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int
    params_loss: str
    params_optimizer: str
    params_learning_rate: float
    params_metrics: list


## configuration manager in src config

In [10]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [11]:
import dagshub
dagshub.init(repo_owner='MannShrestha', repo_name='CT-Scan-Lung-Cancer-Classification', mlflow=True)


Accessing as MannShrestha

Initialized MLflow to track repo "MannShrestha/CT-Scan-Lung-Cancer-Classification"

Repository MannShrestha/CT-Scan-Lung-Cancer-Classification initialized!

In [12]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        evaluation = self.config.evaluation

        evaluation_config = EvaluationConfig(
            path_of_model=Path(evaluation.path_of_model),
            training_data=Path(self.config.data_ingestion.unzip_dir)/"Chest-CT-Scan-data",
            mlflow_uri=evaluation.mlflow_uri,
            all_params=dict(self.params),
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            params_loss=self.params.LOSS,
            params_optimizer=self.params.OPTIMIZER,
            params_learning_rate=self.params.LEARNING_RATE,
            params_metrics=self.params.METRICS
        )

        return evaluation_config



## Model Evaluation Components

In [13]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

c:\Users\USER\anaconda3\envs\cls\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# class Evaluation:
#     def __init__(self, config: EvaluationConfig):
#         self.config = config

    
#     def _valid_generator(self):

#         datagenerator_kwargs = dict(
#             rescale = 1./255,
#             validation_split=0.30
#         )

#         dataflow_kwargs = dict(
#             target_size=self.config.params_image_size[:-1],
#             batch_size=self.config.params_batch_size,
#             interpolation="bilinear"
#         )

#         valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
#             **datagenerator_kwargs
#         )

#         self.valid_generator = valid_datagenerator.flow_from_directory(
#             directory=self.config.training_data,
#             subset="validation",
#             shuffle=False,
#             **dataflow_kwargs
#         )


#     @staticmethod
#     def load_model(path: Path) -> tf.keras.Model:
#         return tf.keras.models.load_model(path)
    

#     def evaluation(self):
#         self.model = self.load_model(self.config.path_of_model)
#         self._valid_generator()
#         self.score = model.evaluate(self.valid_generator)
#         self.save_score()

#     def save_score(self):
#         scores = {"loss": self.score[0], "accuracy": self.score[1]}
#         save_json(path=Path("scores.json"), data=scores)

    
#     def log_into_mlflow(self):
#         mlflow.set_registry_uri(self.config.mlflow_uri)
#         tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
#         with mlflow.start_run():
#             mlflow.log_params(self.config.all_params)
#             mlflow.log_metrics(
#                 {"loss": self.score[0], "accuracy": self.score[1]}
#             )
#             # Model registry does not work with file store
#             if tracking_url_type_store != "file":

#                 # Register the model
#                 # There are other ways to use the Model Registry, which depends on the use case,
#                 # please refer to the doc for more information:
#                 # https://mlflow.org/docs/latest/model-registry.html#api-workflow
#                 mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
#             else:
#                 mlflow.keras.log_model(self.model, "model")


In [15]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config
        self.model = None
        self.valid_generator = None
        self.score = {}

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1.0 / 255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = (
            tf.keras.preprocessing.image.ImageDataGenerator(
                **datagenerator_kwargs
            )
        )

        self.valid_generator = (
            valid_datagenerator.flow_from_directory(
                directory=self.config.training_data,
                subset="validation",
                shuffle=False,
                **dataflow_kwargs
            )
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(
            self.config.path_of_model
        )

        self._valid_generator()

        # Evaluate the model and return named metrics
        self.score = self.model.evaluate(
            self.valid_generator,
            return_dict=True,
            verbose=1
        )

        self.save_score()

        return self.score

    def save_score(self):
        scores = {
            metric: float(value)
            for metric, value in self.score.items()
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_registry_uri(self.config.mlflow_uri)

        with mlflow.start_run():
            # MLflow parameters must be serializable as strings
            params = {
                key: str(value)
                for key, value in self.config.all_params.items()
            }

            mlflow.log_params(params)

            # Log all evaluation metrics by name
            mlflow.log_metrics({
                metric: float(value)
                for metric, value in self.score.items()
            })

            tracking_url_type_store = urlparse(
                mlflow.get_tracking_uri()
            ).scheme

            if tracking_url_type_store != "file":
                mlflow.keras.log_model(
                    self.model,
                    name="model",
                    registered_model_name="VGG16Model"
                )
            else:
                mlflow.keras.log_model(
                    self.model,
                    name="model"
                )

## Model Evaluation Pipeline

In [16]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(config=eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
    raise e


Found 68 images belonging to 2 classes.
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 645ms/step - accuracy: 0.7794 - loss: 0.3996 - precision: 0.7794 - recall: 0.7794


2026/10/21 01:57:34 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/10/21 01:58:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 2
Created version '2' of model 'VGG16Model'.


🏃 View run bedecked-snipe-673 at: https://dagshub.com/MannShrestha/CT-Scan-Lung-Cancer-Classification.mlflow/#/experiments/0/runs/6caa2ae9e7c045909c5d44fff8a5375d
🧪 View experiment at: https://dagshub.com/MannShrestha/CT-Scan-Lung-Cancer-Classification.mlflow/#/experiments/0
